In [1]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)

import pandas as pd

import matplotlib.pyplot as plt

In [2]:
import torch

In [3]:
lsoa_id = sorted(pd.read_csv("../Output/LSOA21CD.csv")["LSOA21CD"].tolist())

embeddings = np.load('../Model/emb_all-MiniLM-L6-v2.npy')
description_topic = pd.read_csv('../Output/NLP Output/feature_extraction_origin.csv')

In [4]:
sbert_df = pd.DataFrame(embeddings)

In [5]:
import ast

description_topic['prob_vector'] = description_topic['probabilities'].apply(ast.literal_eval)

In [6]:
topic_matrix = np.vstack(description_topic['prob_vector'].values)
topic_df = pd.DataFrame(topic_matrix, columns=[f'topic_{i}' for i in range(topic_matrix.shape[1])])

In [7]:
df_all = pd.concat([description_topic, topic_df, sbert_df], axis=1)
df_nlp = df_all.drop(columns=['description', 'topic', 'probabilities', 'prob_vector'])

In [8]:
df_nlp

,LSOA21CD,year,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,...,374,375,376,377,378,379,380,381,382,383
0,E01033084,2002,2.446652e-04,2.102272e-04,1.856370e-04,1.605959e-04,2.068002e-04,9.624113e-05,8.893767e-05,1.276421e-04,...,0.065126,0.006564,0.044565,-0.049141,0.077937,0.043583,-0.106447,0.013555,-0.028803,0.000986
1,E01002676,2002,3.113932e-04,3.376181e-04,3.395465e-04,4.959566e-04,2.979200e-04,6.141201e-04,2.503583e-03,2.650973e-04,...,0.082411,0.065045,0.019736,0.052101,0.014027,0.023311,0.080643,-0.023715,0.038558,0.048565
2,E01002662,2002,2.018424e-306,2.240213e-306,2.140056e-306,9.606314e-307,1.152876e-306,6.933691e-307,5.744552e-307,6.489069e-307,...,0.014257,0.066394,-0.035941,0.029888,0.066311,0.028401,-0.005959,-0.010439,0.014559,-0.005935
3,E01002687,2002,2.227479e-01,1.119299e-03,9.470741e-04,5.334149e-04,9.124164e-04,3.766496e-04,3.579308e-04,4.531367e-04,...,0.031683,-0.035986,-0.031902,-0.043661,0.033311,0.045294,-0.027301,-0.041521,0.019800,-0.013135
4,E01002649,2002,3.546263e-04,3.606364e-04,3.005669e-04,1.606891e-04,3.023880e-04,1.098932e-04,9.685141e-05,1.338884e-04,...,0.037091,0.002158,-0.088090,-0.048079,0.015366,0.044858,-0.056914,-0.063085,0.002108,-0.064479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113933,E01000638,2021,1.225738e-306,9.051317e-307,1.095371e-306,9.655573e-307,7.627033e-306,5.866553e-307,4.920145e-307,9.023499e-307,...,0.058522,0.015918,-0.029230,-0.057205,0.039704,0.027925,-0.019981,-0.024097,-0.020959,0.033360
113934,E01000488,2021,9.973086e-05,1.053339e-04,1.090802e-04,9.915776e-05,1.100874e-04,8.110994e-05,7.927814e-05,1.978663e-04,...,-0.011409,0.051760,0.020795,0.031972,0.061616,0.037672,0.003229,-0.034606,0.068963,0.014466
113935,E01034149,2021,1.196073e-03,1.633515e-03,1.360441e-01,7.625211e-04,1.007015e-03,5.737693e-04,4.782645e-04,6.190871e-04,...,0.055119,0.048625,0.000119,0.002588,0.073450,0.006472,-0.023461,-0.054904,-0.022584,0.032515
113936,E01000643,2021,2.030840e-04,2.164380e-04,2.427999e-04,9.816334e-04,2.267625e-04,1.281264e-03,4.618953e-04,1.785887e-04,...,0.038901,0.056045,0.031970,-0.052313,0.078986,0.006388,-0.002567,-0.048896,0.020874,0.036868


In [9]:
from sklearn.preprocessing import normalize, StandardScaler

# === Basic checks ===
required_cols = {"LSOA21CD", "year"}
missing = required_cols - set(df_nlp.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")
df_nlp["year"] = df_nlp["year"].astype(int)

# === Identify columns ===
embedding_cols = [c for c in df_nlp.columns if str(c).isdigit()]
topic_cols = [c for c in df_nlp.columns if str(c).startswith("topic_")]
vector_cols = topic_cols + embedding_cols
if not embedding_cols:
    raise ValueError("No embedding columns (pure numeric column names) found.")

# === L2-normalize embeddings row-wise ===
df_nlp.loc[:, embedding_cols] = normalize(
    df_nlp[embedding_cols].to_numpy(dtype=np.float32, copy=True),
    norm="l2",
    axis=1
)

In [10]:
# === Aggregate + fill full LSOA×year grid + generate has_text ===
def aggregate_by_lsoaxyear_fill_with_mask(
    dfin: pd.DataFrame,
    years: list[int],
    lsoa_order: list[str],
    cols_to_mean: list[str]
) -> pd.DataFrame:
    """
    Returns a complete LSOAxyear grid, including:
      - Mean-pooled topic/embedding values
      - has_text: whether this (LSOA, year) had original text records (1/0)
    """
    # Filter for the selected years
    dfx = dfin[dfin["year"].isin(years)].copy()

    # Count the number of text records for each (LSOA, year)
    cnt = (
        dfx.groupby(["LSOA21CD", "year"], as_index=False)
           .size()
           .rename(columns={"size": "n_texts"})
    )

    # Aggregate by mean
    if dfx.empty:
        # If no data for this period, create empty skeleton with zeros
        grid = pd.MultiIndex.from_product(
            [lsoa_order, years], names=["LSOA21CD", "year"]
        ).to_frame(index=False)
        for c in cols_to_mean:
            grid[c] = 0.0
        grid["has_text"] = 0
        return grid

    grouped = (
        dfx.groupby(["LSOA21CD", "year"], as_index=False)[cols_to_mean]
           .mean()
           .reset_index(drop=True)
    )

    # Merge with text counts to create has_text column
    grouped = grouped.merge(cnt, on=["LSOA21CD", "year"], how="left")
    grouped["has_text"] = (grouped["n_texts"] > 0).astype(np.int8)
    grouped = grouped.drop(columns=["n_texts"])

    # Build the full LSOA×year grid
    grid = pd.MultiIndex.from_product(
        [lsoa_order, sorted(years)], names=["LSOA21CD", "year"]
    ).to_frame(index=False)

    # Left-join aggregated values
    out = grid.merge(grouped, on=["LSOA21CD", "year"], how="left")

    # Fill missing numeric values with zeros; has_text with 0
    out[cols_to_mean] = out[cols_to_mean].fillna(0.0).astype(np.float32)
    out["has_text"] = out["has_text"].fillna(0).astype(np.int8)

    # Preserve the original LSOA order
    pos = {code: i for i, code in enumerate(lsoa_order)}
    out["_pos"] = out["LSOA21CD"].map(pos)
    out = (
        out.sort_values(["_pos", "year"], kind="mergesort")
           .drop(columns="_pos")
           .reset_index(drop=True)
    )
    return out

In [11]:
# === Aggregate + fill full LSOA + generate has_text ===
def aggregate_by_lsoa_fill_with_mask(
    dfin: pd.DataFrame,
    years: list[int],
    lsoa_order: list[str],
    cols_to_mean: list[str],
) -> pd.DataFrame:
    """
    Returns a per-LSOA table for the selected period (no per-year dimension):
      - Mean-pooled topic/embedding values across all rows within 'years'
      - has_text: whether this LSOA had any text records in the selected period (1/0)
    Output schema:
      LSOA21CD, <cols_to_mean...>, has_text
    """

    # Filter rows within the selected years
    dfx = dfin[dfin["year"].isin(years)].copy()

    # Count records per LSOA to derive has_text
    cnt = (
        dfx.groupby(["LSOA21CD"], as_index=False)
           .size()
           .rename(columns={"size": "n_texts"})
    )

    # Aggregate by LSOA (mean over the selected period)
    if dfx.empty:
        # If no data for this period, return an all-zero skeleton at LSOA level
        grid = pd.DataFrame({"LSOA21CD": lsoa_order})
        for c in cols_to_mean:
            grid[c] = 0.0
        grid["has_text"] = 0
        return grid

    grouped = (
        dfx.groupby(["LSOA21CD"], as_index=False)[cols_to_mean]
           .mean()
           .reset_index(drop=True)
    )

    # Merge has_text
    grouped = grouped.merge(cnt, on="LSOA21CD", how="left")
    grouped["has_text"] = (grouped["n_texts"] > 0).astype(np.int8)
    grouped = grouped.drop(columns=["n_texts"])

    # Build full LSOA grid and left-join, so missing LSOAs are filled
    grid = pd.DataFrame({"LSOA21CD": lsoa_order})
    out = grid.merge(grouped, on="LSOA21CD", how="left")

    # Fill missing numeric values with zeros; has_text with 0
    out[cols_to_mean] = out[cols_to_mean].fillna(0.0).astype(np.float32)
    out["has_text"] = out["has_text"].fillna(0).astype(np.int8)
    
    # Preserve the original LSOA order
    pos = {code: i for i, code in enumerate(lsoa_order)}
    out["_pos"] = out["LSOA21CD"].map(pos)
    out = (
        out.sort_values(["_pos"], kind="mergesort")
        .drop(columns="_pos")
        .reset_index(drop=True)
    )
    return out

In [12]:
# === Build datasets for both periods ===
years_train   = list(range(2002, 2012))   # 2002–2011
years_predict = list(range(2012, 2022))   # 2012–2021

In [13]:
df_train_full = aggregate_by_lsoaxyear_fill_with_mask(df_nlp, years_train, lsoa_id, vector_cols)
df_predict_full = aggregate_by_lsoaxyear_fill_with_mask(df_nlp, years_predict, lsoa_id, vector_cols)

df_train_full_ = aggregate_by_lsoa_fill_with_mask(df_nlp, years_train, lsoa_id, vector_cols)
df_predict_full_ = aggregate_by_lsoa_fill_with_mask(df_nlp, years_predict, lsoa_id, vector_cols)

C:\Users\wbwha\AppData\Local\Temp\ipykernel_10264\2274660676.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["_pos"] = out["LSOA21CD"].map(pos)
C:\Users\wbwha\AppData\Local\Temp\ipykernel_10264\2274660676.py:59: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["_pos"] = out["LSOA21CD"].map(pos)
C:\Users\wbwha\AppData\Local\Temp\ipykernel_10264\4236256711.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider join

In [14]:
df_train_full_

,LSOA21CD,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,...,375,376,377,378,379,380,381,382,383,has_text
0,E01000001,0.000513,0.000427,0.000490,0.000376,0.002867,0.000226,0.000192,0.000342,0.000193,...,0.003560,-0.027292,-0.004055,0.039660,0.039068,-0.011103,0.018177,-0.029624,0.007897,1
1,E01000002,0.000356,0.000342,0.000416,0.000387,0.002230,0.000374,0.000559,0.000265,0.000275,...,0.025834,-0.030747,-0.016480,-0.019121,0.015883,0.022817,-0.001578,-0.084753,0.006518,1
2,E01000003,0.000428,0.000450,0.000473,0.000218,0.000541,0.000132,0.000112,0.000168,0.000115,...,0.118972,0.027737,-0.069239,-0.052776,0.070683,-0.001631,-0.004397,0.004260,0.016983,1
3,E01000005,0.000374,0.000330,0.000386,0.000267,0.001963,0.000164,0.000136,0.000238,0.000140,...,0.023291,-0.004592,-0.023511,0.003778,0.033694,-0.003991,0.014324,-0.025039,0.015430,1
4,E01000006,0.000501,0.001183,0.000463,0.000286,0.000302,0.000242,0.000355,0.000203,0.000219,...,0.022448,-0.045709,0.038895,0.003162,0.044219,-0.026459,0.008211,-0.029152,-0.003815,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4989,E01035718,0.002129,0.000638,0.002955,0.000464,0.000604,0.000399,0.000759,0.000274,0.000297,...,0.044028,-0.011485,0.001759,0.017849,0.023312,0.027083,0.012786,-0.020738,0.011059,1
4990,E01035719,0.000359,0.000344,0.000383,0.000527,0.001044,0.000700,0.000794,0.000272,0.000454,...,0.005894,-0.032945,-0.035226,-0.006731,0.014609,0.014582,-0.010844,-0.065353,0.003740,1
4991,E01035720,0.000584,0.000680,0.009780,0.000473,0.000555,0.000440,0.000366,0.000313,0.000304,...,0.017800,-0.008981,-0.018133,0.016530,0.013420,0.005792,-0.003399,-0.025357,0.038042,1
4992,E01035721,0.000328,0.000432,0.000593,0.000361,0.000304,0.000394,0.001078,0.000196,0.000290,...,0.036875,-0.031683,-0.011831,0.016485,0.013035,0.058852,-0.002965,-0.011827,0.014838,1


In [15]:
# === Standardize topic columns ===
# Fit and transform only for rows with has_text == 1; keep zeros for has_text == 0
scaler_topic_xgboost_gcn = StandardScaler(with_mean=True, with_std=True)

# Train period: fit using rows that have text
mask_tr_ = df_train_full_["has_text"] == 1
if mask_tr_.any():
    scaler_topic_xgboost_gcn.fit(df_train_full_.loc[mask_tr_, topic_cols])
    df_train_full_.loc[mask_tr_, topic_cols] = scaler_topic_xgboost_gcn.transform(
        df_train_full_.loc[mask_tr_, topic_cols]
    ).astype(np.float32)
# Rows without text remain zero

# Predict period: transform using the same scaler
mask_te_ = df_predict_full_["has_text"] == 1
if mask_te_.any():
    df_predict_full_.loc[mask_te_, topic_cols] = scaler_topic_xgboost_gcn.transform(
        df_predict_full_.loc[mask_te_, topic_cols]
    ).astype(np.float32)
# Rows without text remain zero

print("Train (LSOA) period shape:", df_train_full_.shape)
print("Predict (LSOA) period shape:", df_predict_full_.shape)

# Save scaler for inference stage
# import joblib
# joblib.dump(scaler_topic_xgboost_gcn, "topic_scalerc_xgboost_gcn.pkl")

Train (LSOA) period shape: (4994, 1560)
Predict (LSOA) period shape: (4994, 1560)


In [16]:
for _df in (df_train_full_, df_predict_full_):
    if "has_text" not in _df.columns:
        raise ValueError("has_text column not found. Run the aggregation-with-mask step first.")
    mask_ = _df["has_text"] == 1
    if mask_.any():
        _df.loc[mask_, embedding_cols] = normalize(
            _df.loc[mask_, embedding_cols].to_numpy(dtype=np.float32, copy=True),
            norm="l2",
            axis=1
        ).astype(np.float32)

In [17]:
df_train_full_.to_csv("../Output/Pred Input/NLP_train.csv", index=False)
df_predict_full_.to_csv("../Output/Pred Input/NLP_pred.csv", index=False)

In [18]:
import re

def get_topk_topic_cols(topic_cols, k):
    """Return the topic column names for the first k topics (by numeric suffix)."""
    pat = re.compile(r"^topic_(\d+)$")
    indexed = []
    for c in topic_cols:
        m = pat.match(str(c))
        if m:
            indexed.append((int(m.group(1)), c))
    indexed.sort(key=lambda x: x[0])
    return [c for _, c in indexed[:k]]

def to_tensor_2d(df_lsoa, lsoa_order, feature_cols, include_has_text=False):
    """
    Convert an LSOA-level DataFrame (one row per LSOA) into:
      X: [N, D] float32 features  (if include_has_text=True, D += 1 as the last dim)
      M: [N]    bool mask from 'has_text'
    """
    N = len(lsoa_order)
    D = len(feature_cols) + (1 if include_has_text else 0)

    # align rows to lsoa_order
    sub = df_lsoa.set_index("LSOA21CD").loc[lsoa_order]
    X = np.zeros((N, D), dtype=np.float32)

    feats = sub[feature_cols].to_numpy(dtype=np.float32, copy=True)
    X[:, :len(feature_cols)] = feats

    has_text = sub["has_text"].to_numpy().astype(bool)
    if include_has_text:
        X[:, -1] = has_text.astype(np.float32)

    return torch.from_numpy(X), torch.from_numpy(has_text)

tensor_outputs_ = {}

# Embedding only
tensor_outputs_["embedding_only"] = to_tensor_2d(df_train_full_, lsoa_id, embedding_cols, include_has_text=False)
tensor_outputs_["embedding_only"] = to_tensor_2d(df_predict_full_, lsoa_id, embedding_cols, include_has_text=False)

# All topics
tensor_outputs_["topic_all"] = to_tensor_2d(df_train_full_, lsoa_id, topic_cols, include_has_text=False)
tensor_outputs_["topic_all"] = to_tensor_2d(df_predict_full_, lsoa_id, topic_cols, include_has_text=False)

# Top-K topics
for k in [500, 200, 100, 50]:
    topk_cols = get_topk_topic_cols(topic_cols, k)
    key_train = f"topic_top{k}_train"
    key_pred  = f"topic_top{k}_pred"
    tensor_outputs_[key_train] = to_tensor_2d(df_train_full_, lsoa_id, topk_cols)
    tensor_outputs_[key_pred]  = to_tensor_2d(df_predict_full_, lsoa_id, topk_cols)

for name, (X, mask) in tensor_outputs_.items():
    print(f"{name}: X={tuple(X.shape)}, mask={tuple(mask.shape)}  # N={X.shape[0]}, D={X.shape[1]}")

embedding_only: X=(4994, 384), mask=(4994,)  # N=4994, D=384
topic_all: X=(4994, 1174), mask=(4994,)  # N=4994, D=1174
topic_top500_train: X=(4994, 500), mask=(4994,)  # N=4994, D=500
topic_top500_pred: X=(4994, 500), mask=(4994,)  # N=4994, D=500
topic_top200_train: X=(4994, 200), mask=(4994,)  # N=4994, D=200
topic_top200_pred: X=(4994, 200), mask=(4994,)  # N=4994, D=200
topic_top100_train: X=(4994, 100), mask=(4994,)  # N=4994, D=100
topic_top100_pred: X=(4994, 100), mask=(4994,)  # N=4994, D=100
topic_top50_train: X=(4994, 50), mask=(4994,)  # N=4994, D=50
topic_top50_pred: X=(4994, 50), mask=(4994,)  # N=4994, D=50


In [19]:
# === Standardize topic columns ===
# Fit and transform only for rows with has_text == 1; keep zeros for has_text == 0
scaler_topic_gcnlstm = StandardScaler(with_mean=True, with_std=True)

# Train period: fit using rows that have text
mask_tr = df_train_full["has_text"] == 1
if mask_tr.any():
    scaler_topic_gcnlstm.fit(df_train_full.loc[mask_tr, topic_cols])
    df_train_full.loc[mask_tr, topic_cols] = scaler_topic_gcnlstm.transform(
        df_train_full.loc[mask_tr, topic_cols]
    ).astype(np.float32)
# Rows without text remain zero

# Predict period: transform using the same scaler
mask_te = df_predict_full["has_text"] == 1
if mask_te.any():
    df_predict_full.loc[mask_te, topic_cols] = scaler_topic_gcnlstm.transform(
        df_predict_full.loc[mask_te, topic_cols]
    ).astype(np.float32)
# Rows without text remain zero

print("Train (LSOA x Year) period shape:", df_train_full.shape)
print("Predict (LSOA x Year) period shape:", df_predict_full.shape)

# Save scaler for inference stage
# import joblib
# joblib.dump(scaler_topic_gcnlstm, "topic_scaler_gcnlstm.pkl")

Train (LSOA x Year) period shape: (49940, 1561)
Predict (LSOA x Year) period shape: (49940, 1561)


In [20]:
for _df in (df_train_full, df_predict_full):
    if "has_text" not in _df.columns:
        raise ValueError("has_text column not found. Run the aggregation-with-mask step first.")
    mask = _df["has_text"] == 1
    if mask.any():
        _df.loc[mask, embedding_cols] = normalize(
            _df.loc[mask, embedding_cols].to_numpy(dtype=np.float32, copy=True),
            norm="l2",
            axis=1
        ).astype(np.float32)

In [21]:
def to_tensor_3d(df_period, lsoa_order, years, feature_cols, include_has_text=False):
    """
    Convert a complete LSOA×year grid DataFrame into:
      X: [N, T, D] float32 features
      M: [N, T]    bool mask (has_text)
      years_sorted: list of years (sorted)
    """
    years_sorted = sorted(years)
    N = len(lsoa_order)
    D = len(feature_cols) + (1 if include_has_text else 0)
    T = len(years_sorted)

    X = np.zeros((N, T, D), dtype=np.float32)
    M = np.zeros((N, T), dtype=bool)

    for t, y in enumerate(years_sorted):
        sub = df_period[df_period["year"] == y].set_index("LSOA21CD")

        feats = sub.loc[lsoa_order, feature_cols].to_numpy(dtype=np.float32, copy=True)
        X[:, t, :len(feature_cols)] = feats

        has_text_col = sub.loc[lsoa_order, "has_text"].to_numpy()
        M[:, t] = has_text_col.astype(bool)

        if include_has_text:
            X[:, t, -1] = has_text_col.astype(np.float32)

    return torch.from_numpy(X), torch.from_numpy(M), years_sorted

# === Build different feature sets ===
tensor_outputs = {}

# Embedding only
tensor_outputs["embedding_only_train"] = to_tensor_3d(df_train_full, lsoa_id, years_train, embedding_cols)
tensor_outputs["embedding_only_pred"]  = to_tensor_3d(df_predict_full, lsoa_id, years_predict, embedding_cols)

# All topics
tensor_outputs["topic_all_train"] = to_tensor_3d(df_train_full, lsoa_id, years_train, topic_cols)
tensor_outputs["topic_all_pred"]  = to_tensor_3d(df_predict_full, lsoa_id, years_predict, topic_cols)

# Top-K topics
for k in [500, 200, 100, 50]:
    topk_cols = get_topk_topic_cols(topic_cols, k)
    key_train = f"topic_top{k}_train"
    key_pred  = f"topic_top{k}_pred"
    tensor_outputs[key_train] = to_tensor_3d(df_train_full, lsoa_id, years_train, topk_cols)
    tensor_outputs[key_pred]  = to_tensor_3d(df_predict_full, lsoa_id, years_predict, topk_cols)

for name, (X, mask, years) in tensor_outputs.items():
    print(f"{name}: X={tuple(X.shape)}, mask={tuple(mask.shape)}, years={years[:3]}...{years[-3:]}")

embedding_only_train: X=(4994, 10, 384), mask=(4994, 10), years=[2002, 2003, 2004]...[2009, 2010, 2011]
embedding_only_pred: X=(4994, 10, 384), mask=(4994, 10), years=[2012, 2013, 2014]...[2019, 2020, 2021]
topic_all_train: X=(4994, 10, 1174), mask=(4994, 10), years=[2002, 2003, 2004]...[2009, 2010, 2011]
topic_all_pred: X=(4994, 10, 1174), mask=(4994, 10), years=[2012, 2013, 2014]...[2019, 2020, 2021]
topic_top500_train: X=(4994, 10, 500), mask=(4994, 10), years=[2002, 2003, 2004]...[2009, 2010, 2011]
topic_top500_pred: X=(4994, 10, 500), mask=(4994, 10), years=[2012, 2013, 2014]...[2019, 2020, 2021]
topic_top200_train: X=(4994, 10, 200), mask=(4994, 10), years=[2002, 2003, 2004]...[2009, 2010, 2011]
topic_top200_pred: X=(4994, 10, 200), mask=(4994, 10), years=[2012, 2013, 2014]...[2019, 2020, 2021]
topic_top100_train: X=(4994, 10, 100), mask=(4994, 10), years=[2002, 2003, 2004]...[2009, 2010, 2011]
topic_top100_pred: X=(4994, 10, 100), mask=(4994, 10), years=[2012, 2013, 2014]...[201

In [ ]:
import os

def save_tensor_auto(
    name: str,
    model_name: str,
    X: torch.Tensor,
    mask=None,
    years=None,
    save_dir: str = "../Output/Pred Input",
    save_mask: bool = True,
    save_years: bool = True
):
    """
    Save (X, mask, years) to a .pt file with standardized naming.

    Parameters
    ----------
    name : str
        Key from tensor_outputs, e.g. "topic_top100_train".
    model_name : str
        Model identifier for filename.
    X : torch.Tensor
        Feature tensor, either:
          - 2D: [N, D]
          - 3D: [N, T, D]
    mask : torch.Tensor or None
        Boolean mask tensor [N, T] if 3D, or None for 2D.
    years : list[int] | torch.Tensor | None
        Year list if applicable (3D tensors).
    save_dir : str
        Directory to save .pt file.
    save_mask : bool
        Whether to save mask.
    save_years : bool
        Whether to save years.
    """
    os.makedirs(save_dir, exist_ok=True)

    # File name: name_model.pt
    fname = f"{name}_{model_name}.pt"
    path = os.path.join(save_dir, fname)

    payload = {"X": X}

    # Save mask only if 3D and user wants to save
    if save_mask and mask is not None:
        payload["mask"] = mask

    # Save years only if 3D and user wants to save
    if save_years and years is not None:
        payload["years"] = years

    torch.save(payload, path)
    print(f"Saved tensor to: {path}")
    print(f"  Shape of X: {tuple(X.shape)}")
    if "mask" in payload:
        print(f"   Shape of mask: {tuple(mask.shape)}")
    if "years" in payload:
        print(f"   Years: {years[:3]}...{years[-3:]}" if len(years) > 6 else f"   Years: {years}")

In [ ]:
# === Save everything in tensor_outputs ===
# By default we save X, mask, and years for each entry.
out_dir = "../Output/Pred Input"

for name, (X, mask) in tensor_outputs_.items():
    save_tensor_auto(
        name=name,
        model_name="xgboost_gcn",
        X=X,
        mask=mask,
        years=years,
        save_dir=out_dir,
        save_mask=True,      # set False if you don't want to store mask
        save_years=True     # set False if you don't want to store years
    )

for name, (X, mask, years) in tensor_outputs.items():
    save_tensor_auto(
        name=name,
        model_name="gcnlstm",
        X=X,
        mask=mask,
        years=years,
        save_dir=out_dir,
        save_mask=True,      # set False if you don't want to store mask
        save_years=True     # set False if you don't want to store years
    )

Saved tensor to: ../Output/Pred Input\embedding_only_xgboost_gcn.pt
  Shape of X: (4994, 384)
   Shape of mask: (4994,)
   Years: [2012, 2013, 2014]...[2019, 2020, 2021]
Saved tensor to: ../Output/Pred Input\topic_all_xgboost_gcn.pt
  Shape of X: (4994, 1174)
   Shape of mask: (4994,)
   Years: [2012, 2013, 2014]...[2019, 2020, 2021]
Saved tensor to: ../Output/Pred Input\topic_top500_train_xgboost_gcn.pt
  Shape of X: (4994, 500)
   Shape of mask: (4994,)
   Years: [2012, 2013, 2014]...[2019, 2020, 2021]
Saved tensor to: ../Output/Pred Input\topic_top500_pred_xgboost_gcn.pt
  Shape of X: (4994, 500)
   Shape of mask: (4994,)
   Years: [2012, 2013, 2014]...[2019, 2020, 2021]
Saved tensor to: ../Output/Pred Input\topic_top200_train_xgboost_gcn.pt
  Shape of X: (4994, 200)
   Shape of mask: (4994,)
   Years: [2012, 2013, 2014]...[2019, 2020, 2021]
Saved tensor to: ../Output/Pred Input\topic_top200_pred_xgboost_gcn.pt
  Shape of X: (4994, 200)
   Shape of mask: (4994,)
   Years: [2012, 201

In [ ]:
Y_5 = pd.read_csv("../Output/Gentrification label_2021_5class.csv").sort_values("LSOA21CD").reset_index(drop=True)
Y_7 = pd.read_csv("../Output/Gentrification label_2021_7class.csv").sort_values("LSOA21CD").reset_index(drop=True)

In [ ]:
label_mapping_5 = {
    "Class 0 - Affluent and Middle Class": 0,
    "Class 1 - Stable Low-income": 1,
    "Class 2 - Ongoing Displacement": 2,
    "Class 3 - At Risk of Gentrification": 3,
    "Class 4 - Ongoing Gentrification": 4
}

Y_5["label_int"] = Y_5["gentrification_class"].map(label_mapping_5)

label_dict_5 = dict(zip(Y_5["LSOA21CD"], Y_5["label_int"]))

y_train_5 = torch.full((len(lsoa_id),), -1, dtype=torch.long)

for i, lsoa in enumerate(lsoa_id):
    if lsoa in label_dict_5:
        y_train_5[i] = label_dict_5[lsoa]

In [ ]:
label_mapping_7 = {
    "Class 0 - Stable Affluent and Moderate-income": 0,
    "Class 1 - Upgrading or Gentrified": 1,
    "Class 2 - Stable Low-income": 2,
    "Class 3 - Economically Decline": 3,
    "Class 4 - Ongoing Displacement": 4,
    "Class 5 - At Risk of Gentrification": 5,
    "Class 6 - Ongoing Gentrification": 6
}

Y_7["label_int"] = Y_7["gentrification_class"].map(label_mapping_7)

label_dict_7 = dict(zip(Y_7["LSOA21CD"], Y_7["label_int"]))

y_train_7 = torch.full((len(lsoa_id),), -1, dtype=torch.long)

for i, lsoa in enumerate(lsoa_id):
    if lsoa in label_dict_7:
        y_train_7[i] = label_dict_7[lsoa]

In [ ]:
torch.save(y_train_5, "../Output/Pred Input/y_train_5.pt")
torch.save(y_train_7, "../Output/Pred Input/y_train_7.pt")